# 🚀 Train DeBERTa v3 for Resume NER - Team Workflow

This notebook is designed for **team collaboration** where multiple employees annotate resumes using Label Studio.

**Workflow:**
1. Each team member exports their Label Studio annotations as JSON
2. Upload all JSON files to this notebook
3. Automatically merge and deduplicate the data
4. Train DeBERTa v3 model with GPU acceleration
5. Download the trained model

**Labels:**
- Work Experience: COMPANY, ROLE, DATE_START, DATE_END, LOCATION, CLIENT
- Education: DEGREE, INSTITUTION, FIELD, GRADE, EDU_YEAR_START, EDU_YEAR_END

**GPU Setup:** Runtime → Change runtime type → GPU (T4 or better)

## 1️⃣ Check GPU Availability

In [ ]:
import torch

if torch.cuda.is_available():
    print("✅ GPU is available!")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  GPU not available. Training will be slow!")
    print("   Go to: Runtime → Change runtime type → GPU")

## 2️⃣ Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate evaluate seqeval
print("✅ Dependencies installed")

## 3️⃣ Upload Label Studio JSON Files from Team

**Instructions:**
- Collect all JSON export files from your team members
- Upload them all at once (you can select multiple files)
- Files should be named like: `datalabel.json`, `datalabel1.json`, etc.

In [ ]:
from google.colab import files
import json
import os

print("📤 Upload ALL Label Studio JSON files from your team...")
print("   (You can select multiple files at once)\n")

uploaded = files.upload()

print(f"\n✅ Uploaded {len(uploaded)} files:")
for filename in uploaded.keys():
    size_kb = len(uploaded[filename]) / 1024
    print(f"  - {filename} ({size_kb:.1f} KB)")

## 4️⃣ Merge All JSON Files & Remove Duplicates

In [ ]:
from collections import defaultdict

def load_custom_json(file_path):
    """Load custom JSON format with 'label' field."""
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if isinstance(data, dict):
        return [data]
    return data

def merge_custom_json_files(file_list, remove_duplicates=True):
    """Merge multiple custom JSON files."""
    all_examples = []
    seen_texts = set()
    stats = defaultdict(int)
    
    for file_path in file_list:
        print(f"\n📂 Processing: {file_path}")
        
        try:
            examples = load_custom_json(file_path)
            file_count = 0
            
            for example in examples:
                text = example.get('text', '')
                
                if not text:
                    stats['skipped_no_text'] += 1
                    continue
                
                labels = example.get('label', [])
                if not labels or len(labels) == 0:
                    stats['skipped_no_annotations'] += 1
                    continue
                
                if remove_duplicates:
                    if text in seen_texts:
                        stats['duplicates_removed'] += 1
                        continue
                    seen_texts.add(text)
                
                converted_example = {
                    'text': text,
                    'entities': labels
                }
                
                all_examples.append(converted_example)
                file_count += 1
            
            stats[f'from_{os.path.basename(file_path)}'] = file_count
            print(f"  ✅ Added {file_count} examples")
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
            stats['errors'] += 1
    
    return all_examples, dict(stats)

json_files = [f for f in uploaded.keys() if f.endswith('.json')]

print(f"\n🔄 Merging {len(json_files)} JSON files...")
print("=" * 50)

merged_examples, merge_stats = merge_custom_json_files(json_files, remove_duplicates=True)

print("\n" + "=" * 50)
print(f"\n✅ MERGE COMPLETE!")
print(f"\n📊 Summary:")
print(f"  Total examples: {len(merged_examples)}")
print(f"\n📈 Merge Statistics:")
for key, value in merge_stats.items():
    print(f"  {key}: {value}")

## 5️⃣ Analyze Dataset Quality

In [ ]:
def analyze_dataset(examples):
    """Analyze the merged dataset."""
    label_counts = defaultdict(int)
    total_entities = 0
    total_text_length = 0
    
    for example in examples:
        text = example.get('text', '')
        total_text_length += len(text)
        
        entities = example.get('entities', [])
        for entity in entities:
            if 'labels' in entity:
                for label in entity['labels']:
                    label_counts[label] += 1
                    total_entities += 1
    
    return {
        'total_examples': len(examples),
        'total_entities': total_entities,
        'avg_entities_per_example': total_entities / len(examples) if examples else 0,
        'avg_text_length': total_text_length / len(examples) if examples else 0,
        'label_counts': dict(label_counts)
    }

stats = analyze_dataset(merged_examples)

print("📊 Dataset Analysis:")
print(f"  Total examples: {stats['total_examples']}")
print(f"  Total entities: {stats['total_entities']}")
print(f"  Avg entities per example: {stats['avg_entities_per_example']:.2f}")
print(f"  Avg text length: {stats['avg_text_length']:.0f} chars")
print(f"\n  Label distribution:")
for label, count in sorted(stats['label_counts'].items(), key=lambda x: x[1], reverse=True):
    percentage = (count/stats['total_entities']*100) if stats['total_entities'] > 0 else 0
    print(f"    {label:20s}: {count:5d} ({percentage:.1f}%)")

## 6️⃣ Define Labels & Convert to NER Format

In [ ]:
LABELS = [
    "O",
    "B-COMPANY", "I-COMPANY",
    "B-CLIENT", "I-CLIENT",
    "B-ROLE", "I-ROLE",
    "B-LOCATION", "I-LOCATION",
    "B-DATE_START", "I-DATE_START",
    "B-DATE_END", "I-DATE_END",
    "B-DEGREE", "I-DEGREE",
    "B-FIELD", "I-FIELD",
    "B-INSTITUTION", "I-INSTITUTION",
    "B-EDU_YEAR_START", "I-EDU_YEAR_START",
    "B-EDU_YEAR_END", "I-EDU_YEAR_END",
    "B-GRADE", "I-GRADE"
]

label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for idx, label in enumerate(LABELS)}

print(f"📊 Total labels: {len(LABELS)}")

In [ ]:
def convert_to_ner_format(examples):
    """Convert custom format to NER format."""
    ner_examples = []
    
    for item in examples:
        text = item.get('text', '')
        if not text:
            continue
        
        entities = item.get('entities', [])
        if not entities:
            continue
        
        valid_entities = []
        for entity in entities:
            if 'start' in entity and 'end' in entity and 'labels' in entity:
                valid_entities.append({
                    'start': entity['start'],
                    'end': entity['end'],
                    'label': entity['labels'][0]
                })
        
        if valid_entities:
            ner_examples.append({
                'text': text,
                'entities': sorted(valid_entities, key=lambda x: x['start'])
            })
    
    return ner_examples

ner_examples = convert_to_ner_format(merged_examples)

print(f"✅ Converted {len(ner_examples)} examples to NER format")
if ner_examples:
    sample = ner_examples[0]
    print(f"\nSample:")
    print(f"Text: {sample['text'][:100]}...")
    print(f"Entities: {sample['entities'][:3]}")

## 7️⃣ Create Token-Level BIO Tags

In [ ]:
def create_bio_tags(text, entities):
    """Convert character-level entities to token-level BIO tags."""
    tokens = text.split()
    tags = ['O'] * len(tokens)
    
    char_to_token = {}
    char_pos = 0
    for token_idx, token in enumerate(tokens):
        for i in range(len(token)):
            char_to_token[char_pos + i] = token_idx
        char_pos += len(token) + 1
    
    for entity in entities:
        start_token = char_to_token.get(entity['start'])
        end_token = char_to_token.get(entity['end'] - 1)
        
        if start_token is not None and end_token is not None:
            label = entity['label']
            tags[start_token] = f"B-{label}"
            for i in range(start_token + 1, end_token + 1):
                if i < len(tags):
                    tags[i] = f"I-{label}"
    
    return tokens, tags

tokenized_examples = []
for example in ner_examples:
    tokens, tags = create_bio_tags(example['text'], example['entities'])
    tokenized_examples.append({
        'tokens': tokens,
        'ner_tags': tags
    })

print(f"✅ Created {len(tokenized_examples)} tokenized examples")

## 8️⃣ Create Train/Test Split

In [ ]:
from datasets import Dataset, DatasetDict
import random

random.seed(42)
random.shuffle(tokenized_examples)
split_idx = int(len(tokenized_examples) * 0.9)

train_data = tokenized_examples[:split_idx]
test_data = tokenized_examples[split_idx:]

train_dataset = Dataset.from_dict({
    'tokens': [ex['tokens'] for ex in train_data],
    'ner_tags': [[label2id.get(tag, 0) for tag in ex['ner_tags']] for ex in train_data]
})

test_dataset = Dataset.from_dict({
    'tokens': [ex['tokens'] for ex in test_data],
    'ner_tags': [[label2id.get(tag, 0) for tag in ex['ner_tags']] for ex in test_data]
})

dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print(f"📊 Dataset Split:")
print(f"  Train: {len(train_dataset)} examples")
print(f"  Test:  {len(test_dataset)} examples")
print(f"\n{dataset}")

## 9️⃣ Load DeBERTa v3 Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "microsoft/deberta-v3-base"

print(f"📥 Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"✅ Model loaded on {device}")
print(f"📊 Model size: {model.num_parameters() / 1e6:.1f}M parameters")

## 🔟 Tokenize Dataset

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )
    
    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

print("🔄 Tokenizing dataset...")

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset['train'].column_names
)

print("✅ Dataset tokenized")

## 1️⃣1️⃣ Setup Training

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForTokenClassification
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    predictions = np.argmax(predictions, axis=2)
    
    true_labels = [[LABELS[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [LABELS[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

training_args = TrainingArguments(
    output_dir="./resume-ner-deberta",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=2,
    warmup_steps=500,
    save_total_limit=2,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Trainer configured")
print(f"🚀 Training on: {device.upper()}")

## 1️⃣2️⃣ Train the Model 🚀

In [ ]:
print("🚀 Starting training...\n")
print("=" * 70)

trainer.train()

print("\n" + "=" * 70)
print("\n✅ Training completed!")

## 1️⃣3️⃣ Evaluate Model

In [ ]:
results = trainer.evaluate()

print("\n📊 Final Test Results:")
print("=" * 50)
print(f"  Precision: {results['eval_precision']:.4f}")
print(f"  Recall:    {results['eval_recall']:.4f}")
print(f"  F1 Score:  {results['eval_f1']:.4f}")
print(f"  Accuracy:  {results['eval_accuracy']:.4f}")
print("=" * 50)

## 1️⃣4️⃣ Save & Download Model

In [ ]:
output_dir = "./resume-ner-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

with open(f"{output_dir}/label_mappings.json", "w") as f:
    json.dump({
        "labels": LABELS,
        "label_to_id": label2id,
        "id_to_label": id2label
    }, f, indent=2)

print(f"✅ Model saved to {output_dir}")

print("\n📦 Creating zip file...")
!zip -r resume-ner-final.zip resume-ner-final/

print("\n📥 Downloading model...")
files.download('resume-ner-final.zip')

print("\n✅ Model downloaded!")
print("\n📝 Next steps:")
print("1. Extract the zip file")
print("2. Copy to: ai-service/models/resume-ner-deberta/")
print("3. Restart your AI service")
print("4. Test with your resume parser!")
print("\n🎉 Training complete!")